# PROJETO DATATHON - ONG PASSOS MAGICOS

Notebook para limpeza e padronização dos dados da planilha Excel com os dados pedagógicos dos alunos da ONG Passos Mágicos


In [ ]:
import pandas as pd
import numpy as np
!pip install xlsxwriter

In [2]:
# 1. Definição do path do arquivo de dados
url_excel = "https://raw.githubusercontent.com/LuciAguiar/DataThon/main/BASE_DATATHON.xlsx"


In [ ]:
# 2. Leitura do arquivo Excel
# Usamos sheet_name=None para carregar TODAS as abas de uma única vez.
# Isso retorna um dicionário onde a 'chave' é o nome da aba e o 'valor' é o DataFrame.
dict_abas = pd.read_excel(url_excel, sheet_name=None)

# Vamos visualizar quais foram as abas importadas
nomes_das_abas = list(dict_abas.keys())
print(f"Abas identificadas na planilha: {nomes_das_abas}\n")

# 3. Criação de um dataset (DataFrame) para cada uma das abas
# Mapeamos os DataFrames com base na ordem ou no nome exato das abas
df_aba1 = dict_abas[nomes_das_abas[0]]
df_aba2 = dict_abas[nomes_das_abas[1]]
df_aba3 = dict_abas[nomes_das_abas[2]]

# 4. Verificação dos Datasets carregados
# Exibindo as 5 primeiras linhas de cada um para garantir o sucesso da operação
print(f"--- Dataset 1 (Aba: {nomes_das_abas[0]}) ---")
display(df_aba1.head())

print(f"\n--- Dataset 2 (Aba: {nomes_das_abas[1]}) ---")
display(df_aba2.head())

print(f"\n--- Dataset 3 (Aba: {nomes_das_abas[2]}) ---")
display(df_aba3.head())

In [ ]:
# Remove espaços, caracteres especiais e coloca em maiúsculo em uma única linha por dataset
df_aba1.columns = df_aba1.columns.str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
df_aba2.columns = df_aba2.columns.str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
df_aba3.columns = df_aba3.columns.str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)

# Verificar o resultado
print("Colunas da Aba 1:", df_aba1.columns.tolist())
print("Colunas da Aba 2:", df_aba2.columns.tolist())
print("Colunas da Aba 3:", df_aba3.columns.tolist())

In [ ]:
# Ordenar cada dataset de forma crescente pelo RA e redefinir o índice
df_aba1 = df_aba1.sort_values(by='RA', ignore_index=True)
df_aba2 = df_aba2.sort_values(by='RA', ignore_index=True)
df_aba3 = df_aba3.sort_values(by='RA', ignore_index=True)

In [ ]:
print(f"--- Dataset 1 (Aba: {nomes_das_abas[0]}) ---")
display(df_aba1.head())

print(f"\n--- Dataset 2 (Aba: {nomes_das_abas[1]}) ---")
display(df_aba2.head())

print(f"\n--- Dataset 3 (Aba: {nomes_das_abas[2]}) ---")
display(df_aba3.head())

TRATAMENTO DA COLUNA RA
========================
Verifica se há linhas duplicadas por RA, com o objetivo de tranformar  esta coluna como chave para fazer JOIN entre os datasets

In [ ]:
# Retorna um dicionário mostrando a quantidade de RAs duplicados em cada aba
print({nome: df['RA'].duplicated().sum() for nome, df in dict_abas.items()})

Remove o texto fixo "RA-" da coluna RA e transforma a coluna em numerica

In [ ]:
for nome_aba in dict_abas:
    # 1. Remover o prefixo "RA-" (substituindo por texto vazio)
    dict_abas[nome_aba]['RA'] = dict_abas[nome_aba]['RA'].str.replace('RA-', '', regex=False)

    # 2. Converter a coluna de string (object) para numérico (inteiro)
    dict_abas[nome_aba]['RA'] = pd.to_numeric(dict_abas[nome_aba]['RA'])

# Atualizar as variáveis individuais com os dados já numéricos
df_aba1 = dict_abas[list(dict_abas.keys())[0]]
df_aba2 = dict_abas[list(dict_abas.keys())[1]]
df_aba3 = dict_abas[list(dict_abas.keys())[2]]

# Verificar se a conversão funcionou (o tipo deve aparecer como int64 ou int32)
for nome, df in dict_abas.items():
    print(f"Tipo de dado do RA na aba {nome}: {df['RA'].dtype}")

TRATAMENTO DA COLUNA FASE
=========================

In [ ]:
# Iterar por todos os separadores e mostrar os valores únicos da coluna 'FASE'
for nome_aba, df in dict_abas.items():

    if 'FASE' in df.columns:
        valores_unicos = df['FASE'].unique()
        contagem = df['FASE'].value_counts()

        print(f"📊 Separador (Aba): {nome_aba}")
        print(f"➡️ Valores absolutos únicos: {valores_unicos}")
        print(f"🔢 Contagem de registos por valor:\n{contagem}")
        print("-" * 50)
    else:
        print(f"⚠️ O separador '{nome_aba}' não contém uma coluna chamada 'fase'.")
        print("-" * 50)

Padronização dos dados da coluna FASE, através da criação da coluna FASE_ESCOLAR, transformando todos os dados em numérico

In [ ]:
# 1. Criar a nova coluna FASE_ESCOLAR como cópia da original FASE
df_aba1['FASE_ESCOLAR'] = df_aba1['FASE']
df_aba2['FASE_ESCOLAR'] = df_aba2['FASE']
df_aba3['FASE_ESCOLAR'] = df_aba3['FASE']

# 2. Tratamento no df_aba2 (PEDE2023)
# Substitui 'ALFA' por '0', remove o texto 'FASE ' deixando só o número, e converte para numérico
df_aba2['FASE_ESCOLAR'] = df_aba2['FASE_ESCOLAR'].astype(str).str.replace('ALFA', '0', regex=False)
df_aba2['FASE_ESCOLAR'] = df_aba2['FASE_ESCOLAR'].str.replace(r'\D', '', regex=True)
df_aba2['FASE_ESCOLAR'] = pd.to_numeric(df_aba2['FASE_ESCOLAR'])

# 3. Tratamento no df_aba3 (PEDE2024)
# O mesmo processo lida perfeitamente com os sufixos (1A, 2B, etc.) e o valor solto '9'
df_aba3['FASE_ESCOLAR'] = df_aba3['FASE_ESCOLAR'].astype(str).str.replace('ALFA', '0', regex=False)
df_aba3['FASE_ESCOLAR'] = df_aba3['FASE_ESCOLAR'].str.replace(r'\D', '', regex=True)
df_aba3['FASE_ESCOLAR'] = pd.to_numeric(df_aba3['FASE_ESCOLAR'])

# 4. Tratamento no df_aba1 (PEDE2022)
# Como a aba 1 já está correta, garantimos apenas a conversão para o tipo numérico padrão
df_aba1['FASE_ESCOLAR'] = pd.to_numeric(df_aba1['FASE_ESCOLAR'])

# --- Validação dos Resultados ---
# Imprimimos os valores únicos absolutos e ordenados para confirmar a padronização
print("Valores FASE_ESCOLAR df_aba1:", sorted(df_aba1['FASE_ESCOLAR'].unique()))
print("Valores FASE_ESCOLAR df_aba2:", sorted(df_aba2['FASE_ESCOLAR'].unique()))
print("Valores FASE_ESCOLAR df_aba3:", sorted(df_aba3['FASE_ESCOLAR'].unique()))

In [ ]:
# Iterar por todos os separadores e mostrar os valores únicos da coluna 'FASE'
for nome_aba, df in dict_abas.items():

    if 'FASE_ESCOLAR' in df.columns:
        valores_unicos = df['FASE_ESCOLAR'].unique()
        contagem = df['FASE_ESCOLAR'].value_counts().sort_index()

        print(f"📊 Separador (Aba): {nome_aba}")
        print(f"➡️ Valores absolutos únicos: {valores_unicos}")
        print(f"🔢 Contagem de registos por valor:\n{contagem}")
        print("-" * 50)
    else:
        print(f"⚠️ O separador '{nome_aba}' não contém uma coluna chamada 'fase'.")
        print("-" * 50)

In [ ]:
# Apos criar a coluna FASE_ESCOLAR, elimina a coluna FASE
df_aba1 = df_aba1.drop(columns=['FASE'], errors='ignore')
df_aba2 = df_aba2.drop(columns=['FASE'], errors='ignore')
df_aba3 = df_aba3.drop(columns=['FASE'], errors='ignore')

print("Coluna 'FASE' removida de todos os DataFrames.")

TRATAMENTO DA COLUNA IDADE



In [ ]:
valores_unicos_idade = df_aba1['IDADE22'].unique()
print(f"➡️ Valores absolutos únicos: {valores_unicos_idade}")

valores_unicos_idade = df_aba2['IDADE'].unique()
print(f"➡️ Valores absolutos únicos: {valores_unicos_idade}")

valores_unicos_idade = df_aba3['IDADE'].unique()
print(f"➡️ Valores absolutos únicos: {valores_unicos_idade}")

Identificamos que os dados referentes a idade estão sem padronização entre os 3 anos:

Em 2022, a coluna IDADE22 representa a idade, que é numérica.
Em 2023, a coluna IDADE tem dados numéricos e algumas linhas com datetime, sendo o ano fixo de 1900.
Em 2024, a coluna IDADE está correta.

Estamos partindo da premissa que em 2022, que não apresenta a data de nascimento, apenas o ano de nascimento, a idade apresentada é a idade que o aluno tinha no final do ano.

Para que esta informação fique padronizada nos anos, vamos desprezar os dados da coluna IDADE nos anos de 2023 e 2024 e calcular a idade que o aluno tinha no final do ano, considerando o ano da data de nascimento.

In [ ]:

# 1. Cria a nova coluna IDADEFINAL no dataset referente a 2022 apenas copiando o dado da coluna IDADE22
df_aba1['IDADEFINAL'] = df_aba1['IDADE22']


# 2. Cria a nova coluna IDADEFINAL no dateset referente a 2023, a partir da data de nascimento
# Boa prática: Garantir que a coluna está realmente como datetime
# (o 'coerce' transforma eventuais erros de digitação na planilha em valores nulos 'NaT')
df_aba2['DATADENASC'] = pd.to_datetime(df_aba2['DATADENASC'], errors='coerce')
df_aba2['IDADEFINAL'] = 2023 - df_aba2['DATADENASC'].dt.year

# 3. Criar a nova coluna IDADEFINAL no dateset referente a 2024, a partir da data de nascimento
df_aba3['DATADENASC'] = pd.to_datetime(df_aba3['DATADENASC'], errors='coerce')
df_aba3['IDADEFINAL'] = 2024 - df_aba3['DATADENASC'].dt.year

# 4. Validar visualmente o resultado
display(df_aba1[['IDADE22', 'IDADEFINAL']].head())
display(df_aba2[['DATADENASC', 'IDADEFINAL']].head())
display(df_aba3[['DATADENASC', 'IDADEFINAL']].head())

TRATAMENTO PARA A COLUNA GENERO

Em 2022 o gênero é separado por Menina e Menino. Já nos demais anos, está como Feminino e Masculino. Fizemos harmonização dos dados

In [ ]:
valores_unicos_genero = df_aba1['GNERO'].unique()
print(f"➡️ Valores absolutos únicos: {valores_unicos_genero}")

valores_unicos_genero = df_aba2['GNERO'].unique()
print(f"➡️ Valores absolutos únicos: {valores_unicos_genero}")

valores_unicos_genero = df_aba3['GNERO'].unique()
print(f"➡️ Valores absolutos únicos: {valores_unicos_genero}")

In [ ]:
# Cria a coluna GENERO: se GNERO for 'Menino' recebe 'Masculino', senão recebe 'Feminino'
df_aba1['GNERO'] = np.where(df_aba1['GNERO'] == 'Menino', 'Masculino', 'Feminino')
valores_unicos_genero = df_aba1['GNERO'].unique()
print(f"➡️ Valores absolutos únicos: {valores_unicos_genero}")

In [ ]:
df_aba1 = df_aba1.rename(columns={'GNERO': 'GENERO'})
df_aba2 = df_aba2.rename(columns={'GNERO': 'GENERO'})
df_aba3 = df_aba3.rename(columns={'GNERO': 'GENERO'})

INSIGHT:  Em 2024 há menor quantidade de alunos antigos na instituição

In [ ]:
valores_unicos_anoingresso = np.sort(df_aba1['ANOINGRESSO'].unique())
print(f"➡️ Valores absolutos únicos: {valores_unicos_anoingresso}")

valores_unicos_anoingresso = np.sort(df_aba2['ANOINGRESSO'].unique())
print(f"➡️ Valores absolutos únicos: {valores_unicos_anoingresso}")

valores_unicos_anoingresso = np.sort(df_aba3['ANOINGRESSO'].unique())
print(f"➡️ Valores absolutos únicos: {valores_unicos_anoingresso}")

TRATAMENTO PARA AS COLUNAS PEDRA

Como a intenção é fazer um join das 3 abas da planilha em um unico DATASET, criamos a coluna PEDRAANO e gravamos nela o valor da pedra de cada ano, após limpez e tratamento dos dados

In [ ]:
df_aba1['PEDRAANO'] = df_aba1['PEDRA22']
df_aba2['PEDRAANO'] = df_aba2['PEDRA2023']
df_aba3['PEDRAANO'] = df_aba3['PEDRA2024']

valores_unicos_pedra_ano = np.sort(df_aba1['PEDRAANO'].astype(str).unique())
print(f"➡️ Valores absolutos únicos: {valores_unicos_pedra_ano}")

valores_unicos_pedra_ano = np.sort(df_aba2['PEDRAANO'].astype(str).unique())
print(f"➡️ Valores absolutos únicos: {valores_unicos_pedra_ano}")

valores_unicos_pedra_ano = np.sort(df_aba3['PEDRAANO'].astype(str).unique())
print(f"➡️ Valores absolutos únicos: {valores_unicos_pedra_ano}")

In [ ]:
# Retirando espaços em branco
df_aba1['PEDRAANO'] = df_aba1['PEDRAANO'].astype(str).str.strip()
df_aba2['PEDRAANO'] = df_aba2['PEDRAANO'].astype(str).str.strip()
df_aba3['PEDRAANO'] = df_aba3['PEDRAANO'].astype(str).str.strip()

# Substitindo o valor 'nan' para nulo
df_aba1['PEDRAANO'] = df_aba1['PEDRAANO'].replace('nan', np.nan)
df_aba2['PEDRAANO'] = df_aba2['PEDRAANO'].replace('nan', np.nan)
df_aba3['PEDRAANO'] = df_aba3['PEDRAANO'].replace('nan', np.nan)

# Removendo as linhas com PEDRANO nulas
df_aba1.dropna(subset=['PEDRAANO'], inplace=True)
df_aba2.dropna(subset=['PEDRAANO'], inplace=True)
df_aba3.dropna(subset=['PEDRAANO'], inplace=True)

# Retirando valores indevidos
df_aba2 = df_aba2[df_aba2['PEDRAANO'] != '#DIV/0!']
df_aba3 = df_aba3[df_aba3['PEDRAANO'] != 'INCLUIR']

# Resetando o índice
df_aba1.reset_index(drop=True, inplace=True)
df_aba2.reset_index(drop=True, inplace=True)
df_aba3.reset_index(drop=True, inplace=True)

In [ ]:
df_aba1['PEDRAANO'] = df_aba1['PEDRAANO'].replace('Ágata', 'Agata')
df_aba1['PEDRAANO'] = df_aba1['PEDRAANO'].replace('Topázio', 'Topazio')
df_aba2['PEDRAANO'] = df_aba2['PEDRAANO'].replace('Topázio', 'Topazio')
df_aba3['PEDRAANO'] = df_aba3['PEDRAANO'].replace('Topázio', 'Topazio')

valores_unicos_pedra_ano = np.sort(df_aba1['PEDRAANO'].astype(str).unique())
print(f"➡️ Valores absolutos únicos Aba1: {valores_unicos_pedra_ano}")

valores_unicos_pedra_ano = np.sort(df_aba2['PEDRAANO'].astype(str).unique())
print(f"➡️ Valores absolutos únicos Aba2: {valores_unicos_pedra_ano}")

valores_unicos_pedra_ano = np.sort(df_aba3['PEDRAANO'].astype(str).unique())
print(f"➡️ Valores absolutos únicos Aba3: {valores_unicos_pedra_ano}")

In [ ]:
print(f"--- Dataset 1 (Aba: {nomes_das_abas[0]}) ---")
display(df_aba1.head())

print(f"\n--- Dataset 2 (Aba: {nomes_das_abas[1]}) ---")
display(df_aba2.head())

print(f"\n--- Dataset 3 (Aba: {nomes_das_abas[2]}) ---")
display(df_aba3.head())

TRATAMENTO PARA A COLUNA FASE IDEAL

In [ ]:
valores_unicos_fase_ideal = np.sort(df_aba1['FASEIDEAL'].astype(str).unique())
print(f"➡️ Valores absolutos únicos Aba1: {valores_unicos_fase_ideal}")

In [ ]:
# 1. Garantir que a coluna é do tipo string
df_aba1['FASEIDEAL'] = df_aba1['FASEIDEAL'].astype(str)

# 2. Substituir 'ALFA' por '0' para padronizar com as fases numéricas
df_aba1['FASEIDEAL'] = df_aba1['FASEIDEAL'].str.replace('ALFA', '0', regex=False)

# 3. Extrair a primeira sequência de dígitos e converter para inteiro
# Isso irá extrair o número da fase (e '0' para 'ALFA')
df_aba1['FASEIDEAL'] = df_aba1['FASEIDEAL'].str.extract(r'(\d+)', expand=False).astype(int)

# Exibir os valores únicos transformados para verificação
valores_unicos_fase_ideal_transformados = np.sort(df_aba1['FASEIDEAL'].unique())
print(f"➡️ Valores absolutos únicos da FASEIDEAL (transformados): {valores_unicos_fase_ideal_transformados}")

In [ ]:
# Aplica o mesmo tratamento para df_aba2
df_aba2['FASEIDEAL'] = df_aba2['FASEIDEAL'].astype(str)
df_aba2['FASEIDEAL'] = df_aba2['FASEIDEAL'].str.replace('ALFA', '0', regex=False)
df_aba2['FASEIDEAL'] = df_aba2['FASEIDEAL'].str.extract(r'(\d+)', expand=False).astype(int)

# Exibir os valores únicos transformados para verificação
valores_unicos_fase_ideal_transformados_aba2 = np.sort(df_aba2['FASEIDEAL'].unique())
print(f"➡️ Valores absolutos únicos da FASEIDEAL (transformados) Aba2: {valores_unicos_fase_ideal_transformados_aba2}")

In [ ]:
# Aplica o mesmo tratamento para df_aba3
# Realiza todas as transformações em uma cópia temporária da série para evitar o FutureWarning
temp_faseideal_series = df_aba3['FASEIDEAL'].astype(str)
temp_faseideal_series = temp_faseideal_series.str.replace('ALFA', '0', regex=False)

# Atribui o resultado final, convertido para int, de volta à coluna original
df_aba3.loc[:, 'FASEIDEAL'] = temp_faseideal_series.str.extract(r'(\d+)', expand=False).astype(int)

# Exibir os valores únicos transformados para verificação
valores_unicos_fase_ideal_transformados_aba3 = np.sort(df_aba3['FASEIDEAL'].unique())
print(f"➡️ Valores absolutos únicos da FASEIDEAL (transformados) Aba3: {valores_unicos_fase_ideal_transformados_aba3}")

In [ ]:
valores_unicos_instituicao_aba1 = np.sort(df_aba1['INSTITUIODEENSINO'].astype(str).unique())
print(f"➡️ Valores únicos de INSTITUIUIODEENSINO Aba1: {valores_unicos_instituicao_aba1}")
contagem_instituicao_aba1 = df_aba1['INSTITUIODEENSINO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba1:\n{contagem_instituicao_aba1}\n")

valores_unicos_instituicao_aba2 = np.sort(df_aba2['INSTITUIODEENSINO'].astype(str).unique())
print(f"➡️ Valores únicos de INSTITUIUIODEENSINO Aba2: {valores_unicos_instituicao_aba2}")
contagem_instituicao_aba2 = df_aba2['INSTITUIODEENSINO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba2:\n{contagem_instituicao_aba2}\n")

valores_unicos_instituicao_aba3 = np.sort(df_aba3['INSTITUIODEENSINO'].astype(str).unique())
print(f"➡️ Valores únicos de INSTITUIUIODEENSINO Aba3: {valores_unicos_instituicao_aba3}")
contagem_instituicao_aba3 = df_aba3['INSTITUIODEENSINO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba3:\n{contagem_instituicao_aba3}")

### Padronização da Coluna INSTITUIODEENSINO

In [ ]:
# Função para padronizar a coluna INSTITUIODEENSINO
def padronizar_instituicao(df):
    # 1. Converter para string e preencher NaNs
    df['INSTITUIODEENSINO'] = df['INSTITUIODEENSINO'].astype(str).replace('nan', 'Não Informado')

    # 2. Converter para Title Case (primeira letra de cada palavra em maiúscula)
    df['INSTITUIODEENSINO'] = df['INSTITUIODEENSINO'].apply(lambda x: x.title())

    # 3. Consolidar variações de 'Privada - Programa de Apadrinhamento'
    df['INSTITUIODEENSINO'] = df['INSTITUIODEENSINO'].replace(
        'Privada - Programa De Apadrinhamento',
        'Privada - Programa de Apadrinhamento'
    )

    # 4. Padronizar 'Privada *Parcerias com Bolsa 100%'
    df['INSTITUIODEENSINO'] = df['INSTITUIODEENSINO'].replace(
        'Privada *Parcerias Com Bolsa 100%',
        'Privada - Parcerias com Bolsa 100%'
    )

    return df

# Aplicar a padronização aos três DataFrames
df_aba1 = padronizar_instituicao(df_aba1)
df_aba2 = padronizar_instituicao(df_aba2)
df_aba3 = padronizar_instituicao(df_aba3)

print("Padronização da coluna INSTITUIODEENSINO aplicada aos DataFrames.")

### Verificação Após Padronização

In [ ]:
print("\n--- Resultados para df_aba1 após padronização ---")
valores_unicos_instituicao_aba1 = np.sort(df_aba1['INSTITUIODEENSINO'].astype(str).unique())
print(f"➡️ Valores únicos de INSTITUIODEENSINO Aba1: {valores_unicos_instituicao_aba1}")
contagem_instituicao_aba1 = df_aba1['INSTITUIODEENSINO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba1:\n{contagem_instituicao_aba1}\n")

print("\n--- Resultados para df_aba2 após padronização ---")
valores_unicos_instituicao_aba2 = np.sort(df_aba2['INSTITUIODEENSINO'].astype(str).unique())
print(f"➡️ Valores únicos de INSTITUIODEENSINO Aba2: {valores_unicos_instituicao_aba2}")
contagem_instituicao_aba2 = df_aba2['INSTITUIODEENSINO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba2:\n{contagem_instituicao_aba2}\n")

print("\n--- Resultados para df_aba3 após padronização ---")
valores_unicos_instituicao_aba3 = np.sort(df_aba3['INSTITUIODEENSINO'].astype(str).unique())
print(f"➡️ Valores únicos de INSTITUIODEENSINO Aba3: {valores_unicos_instituicao_aba3}")
contagem_instituicao_aba3 = df_aba3['INSTITUIODEENSINO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba3:\n{contagem_instituicao_aba3}")

### Criação da Coluna TIPOINSTITUICAO

In [ ]:
# Definir as condições e escolhas para a nova coluna TIPOINSTITUICAO
# Lógica corrigida para df_aba1 baseada nas regras explícitas do usuário
conditions_aba1 = [
    df_aba1['INSTITUIODEENSINO'] == 'Escola Pública',
    (df_aba1['INSTITUIODEENSINO'] == 'Escola Jp Ii') | (df_aba1['INSTITUIODEENSINO'] == 'Rede Decisão')
]
choices_aba1 = ['Pública', 'Privada']
# Usar 'Não Classificado' como default temporário para identificar qualquer valor não esperado
df_aba1['TIPOINSTITUICAO'] = np.select(conditions_aba1, choices_aba1, default='Não Classificado')

conditions_aba2 = [
    df_aba2['INSTITUIODEENSINO'] == 'Concluiu O 3º Em',
    df_aba2['INSTITUIODEENSINO'].str.contains('Pública', case=False, na=False)
]
choices_aba2 = ['Concluido', 'Pública']
df_aba2['TIPOINSTITUICAO'] = np.select(conditions_aba2, choices_aba2, default='Privada')

conditions_aba3 = [
    df_aba3['INSTITUIODEENSINO'] == 'Concluiu O 3º Em',
    df_aba3['INSTITUIODEENSINO'].str.contains('Pública', case=False, na=False)
]
choices_aba3 = ['Concluido', 'Pública']
df_aba3['TIPOINSTITUICAO'] = np.select(conditions_aba3, choices_aba3, default='Privada')

print("Coluna TIPOINSTITUICAO recriada com a lógica corrigida em todos os DataFrames.")

### Verificação da Coluna TIPOINSTITUICAO

In [ ]:
print("\n--- Resultados para df_aba1 (TIPOINSTITUICAO) ---")
valores_unicos_tipo_instituicao_aba1 = np.sort(df_aba1['TIPOINSTITUICAO'].unique())
print(f"➡️ Valores únicos de TIPOINSTITUICAO Aba1: {valores_unicos_tipo_instituicao_aba1}")
contagem_tipo_instituicao_aba1 = df_aba1['TIPOINSTITUICAO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba1:\n{contagem_tipo_instituicao_aba1}\n")

print("\n--- Resultados para df_aba2 (TIPOINSTITUICAO) ---")
valores_unicos_tipo_instituicao_aba2 = np.sort(df_aba2['TIPOINSTITUICAO'].unique())
print(f"➡️ Valores únicos de TIPOINSTITUICAO Aba2: {valores_unicos_tipo_instituicao_aba2}")
contagem_tipo_instituicao_aba2 = df_aba2['TIPOINSTITUICAO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba2:\n{contagem_tipo_instituicao_aba2}\n")

print("\n--- Resultados para df_aba3 (TIPOINSTITUICAO) ---")
valores_unicos_tipo_instituicao_aba3 = np.sort(df_aba3['TIPOINSTITUICAO'].unique())
print(f"➡️ Valores únicos de TIPOINSTITUICAO Aba3: {valores_unicos_tipo_instituicao_aba3}")
contagem_tipo_instituicao_aba3 = df_aba3['TIPOINSTITUICAO'].value_counts().sort_index()
print(f"🔢 Contagem de registros por valor em Aba3:\n{contagem_tipo_instituicao_aba3}")

###Removendo os indicadores que foram tratados e não serão utilizados na etapa seguinte de predição

In [ ]:
# Colunas a serem removidas de df_aba1
colunas_remover_aba1 = [
    'NOME',
    'ANONASC',
    'IDADE22',
    'PEDRA22',
    'PEDRA21',
    'PEDRA20',
    'NAV',
    'AVALIADOR1',
    'AVALIADOR2',
    'AVALIADOR3',
    'AVALIADOR4',
    'DESTAQUEIDA',
    'DESTAQUEIEG',
    'DESTAQUEIPV',
    'RECAV1',
    'RECAV2',
    'RECAV3',
    'RECAV4'
    ]
df_aba1 = df_aba1.drop(columns=colunas_remover_aba1)
#df_aba1 = df_aba1.drop(columns=colunas_remover_aba1, errors='ignore')

# Colunas a serem removidas de df_aba2
colunas_remover_aba2 = [
    'PEDRA2023',
    'NOMEANONIMIZADO',
    'DATADENASC',
    'IDADE',
    'PEDRA20',
    'PEDRA21',
    'PEDRA22',
    'PEDRA23',
    'NAV',
    'AVALIADOR1',
    'AVALIADOR2',
    'AVALIADOR3',
    'AVALIADOR4',
    'DESTAQUEIDA',
    'DESTAQUEIEG',
    'DESTAQUEIPV',
    'DESTAQUEIPV1',
    'INDE22',
    'INDE23',
    'RECAV1',
    'RECAV2',
    'RECAV3',
    'RECAV4'
    ]
df_aba2 = df_aba2.drop(columns=colunas_remover_aba2)

# Colunas a serem removidas de df_aba3
colunas_remover_aba3 = [
    'PEDRA2024',
    'NOMEANONIMIZADO',
    'DATADENASC',
    'IDADE',
    'PEDRA20',
    'PEDRA21',
    'PEDRA22',
    'PEDRA23',
    'INDE22',
    'INDE23',
    'NAV',
    'AVALIADOR1',
    'AVALIADOR2',
    'AVALIADOR3',
    'AVALIADOR4',
    'AVALIADOR5',
    'AVALIADOR6',
    'DESTAQUEIDA',
    'DESTAQUEIEG',
    'DESTAQUEIPV',
    'ESCOLA',
    'RECAV1',
    'RECAV2'
]
df_aba3 = df_aba3.drop(columns=colunas_remover_aba3)

print("Colunas removidas dos DataFrames.")

In [ ]:
# Renomeando colunas para padronizar as abas dos 3 anos
df_aba1 = df_aba1.rename(columns={'INDE22': 'INDE', 'DEFAS': 'DEFASAGEM', 'INGLS': 'ING', 'MATEM': 'MAT', 'PORTUG': 'POR'})
df_aba2 = df_aba2.rename(columns={'INDE2023': 'INDE'})
df_aba3 = df_aba3.rename(columns={'INDE2024': 'INDE'})

In [ ]:
print("Colunas atuais em df_aba1:", df_aba1.columns.tolist())
print("Colunas atuais em df_aba2:", df_aba2.columns.tolist())
print("Colunas atuais em df_aba3:", df_aba3.columns.tolist())

TRATAMENTO DAS COLUNAS ATIVOINATIVO E ATIVOINATIVO1

In [ ]:
valores_unicos_ativoinativo = np.sort(df_aba3['ATIVOINATIVO'].unique())
valores_unicos_ativoinativo1 = np.sort(df_aba3['ATIVOINATIVO1'].unique())
print(f"➡️ Valores únicos de ATIVOINATIVO Aba3: {valores_unicos_ativoinativo}")
print(f"➡️ Valores únicos de ATIVOINATIVO1 Aba3: {valores_unicos_ativoinativo1}")

In [ ]:
# Como as 2 colunas tem o mesmo valor e os valores são únicos, podemos removê-las
df_aba3 = df_aba3.drop(columns=['ATIVOINATIVO','ATIVOINATIVO1'])

In [ ]:
print(f"--- Dataset 1 (Aba: {nomes_das_abas[0]}) ---")
display(df_aba1.head())

print(f"\n--- Dataset 2 (Aba: {nomes_das_abas[1]}) ---")
display(df_aba2.head())

print(f"\n--- Dataset 3 (Aba: {nomes_das_abas[2]}) ---")
display(df_aba3.head())

In [ ]:
# Removendo a coluna INSTITUIODEENSO, cujo valor foi agregado na coluna TIPOINSTITUICAO
df_aba1 = df_aba1.drop(columns=['INSTITUIODEENSINO'], errors='ignore')
df_aba2 = df_aba2.drop(columns=['INSTITUIODEENSINO'], errors='ignore')
df_aba3 = df_aba3.drop(columns=['INSTITUIODEENSINO'], errors='ignore')

print("Coluna 'INSTITUIODEENSINO' removida de todos os DataFrames.")

### Verificação de Colunas Consistentes entre DataFrames

Vamos verificar se todos os DataFrames possuem o mesmo conjunto de colunas após os tratamentos. Esta é uma etapa importante para garantir a consistência dos dados antes de qualquer análise ou concatenação.

In [ ]:
colunas_aba1 = set(df_aba1.columns)
colunas_aba2 = set(df_aba2.columns)
colunas_aba3 = set(df_aba3.columns)

# Comparar os conjuntos de colunas
if colunas_aba1 == colunas_aba2 == colunas_aba3:
    print("✅ Todos os DataFrames possuem o mesmo conjunto de colunas.")
    print("Número total de colunas: ", len(colunas_aba1))
    print("Colunas: ", sorted(list(colunas_aba1)))
else:
    print("❌ Os DataFrames NÃO possuem o mesmo conjunto de colunas.")
    if colunas_aba1 != colunas_aba2:
        print("Diferenças entre df_aba1 e df_aba2:")
        print("   Somente em df_aba1:", sorted(list(colunas_aba1 - colunas_aba2)))
        print("   Somente em df_aba2:", sorted(list(colunas_aba2 - colunas_aba1)))
    if colunas_aba1 != colunas_aba3:
        print("Diferenças entre df_aba1 e df_aba3:")
        print("   Somente em df_aba1:", sorted(list(colunas_aba1 - colunas_aba3)))
        print("   Somente em df_aba3:", sorted(list(colunas_aba3 - colunas_aba1)))
    if colunas_aba2 != colunas_aba3:
        print("Diferenças entre df_aba2 e df_aba3:")
        print("   Somente em df_aba2:", sorted(list(colunas_aba2 - colunas_aba3)))
        print("   Somente em df_aba3:", sorted(list(colunas_aba3 - colunas_aba2)))

Adicionando a coluna IPP no dataset df_aba1 para que os 3 datasets fiquem com as mesmas colunas antes de fazer o JOIN

In [ ]:

df_aba1['IPP'] = np.nan

Salvando o arquivo Excel com os dados tratados antes de fazer o JOIN dos datasets


In [ ]:
# Define o nome do arquivo Excel de saída
nome_arquivo_excel = 'dados_corrigidos.xlsx'

# Cria um objeto ExcelWriter
with pd.ExcelWriter(nome_arquivo_excel, engine='xlsxwriter') as writer:
    # Exporta cada DataFrame para uma aba diferente no mesmo arquivo Excel
    df_aba1.to_excel(writer, sheet_name='PEDE2022_Corrigido', index=False)
    df_aba2.to_excel(writer, sheet_name='PEDE2023_Corrigido', index=False)
    df_aba3.to_excel(writer, sheet_name='PEDE2024_Corrigido', index=False)

print(f"Os DataFrames foram exportados com sucesso para '{nome_arquivo_excel}'.")

In [ ]:
df_aba1['ANO'] = 2022
df_aba2['ANO'] = 2023
df_aba3['ANO'] = 2024

### Concatenação dos DataFrames

Com todos os DataFrames agora padronizados e com as mesmas colunas, podemos concatená-los verticalmente para criar um único conjunto de dados consolidado para análise.

In [ ]:
# Concatenar os três DataFrames verticalmente
df_consolidado = pd.concat([df_aba1, df_aba2, df_aba3], ignore_index=True)

print("DataFrame consolidado criado com sucesso!")
print("Exibindo as 5 primeiras linhas do df_consolidado:")
display(df_consolidado.head())

print("\nForma do df_consolidado (linhas, colunas):")
print(df_consolidado.shape)

### Verificação de Duplicatas com RA e ANO como Chave

Agora que o `df_consolidado` foi criado, vamos verificar se a combinação das colunas `RA` e `ANO` forma uma chave única, conforme solicitado. Isso é crucial para garantir a integridade dos dados ao identificar cada aluno em um ano específico.

In [ ]:
# Verificar duplicatas com base nas colunas 'RA' e 'ANO'
duplicatas_ra_ano = df_consolidado[df_consolidado.duplicated(subset=['RA', 'ANO'], keep=False)]

if not duplicatas_ra_ano.empty:
    print("❌ Foram encontradas linhas duplicadas para a combinação de 'RA' e 'ANO':")
    display(duplicatas_ra_ano.sort_values(by=['RA', 'ANO']))
else:
    print("✅ Nenhuma linha duplicada encontrada para a combinação de 'RA' e 'ANO'.")
    print("A combinação de 'RA' e 'ANO' pode ser considerada uma chave única no df_consolidado.")

In [ ]:
print("Tipos de dados das colunas do df_consolidado:")
display(df_consolidado.dtypes)

TRATAMENTO PARA A COLUNA INDE

 Conversão dos valores da coluna 'INDE' no df_consolidado para o tipo float, substituindo as vírgulas por pontos como separadores decimais e arredondando para duas casas decimais

In [ ]:
df_consolidado['INDE'] = df_consolidado['INDE'].astype(str).str.replace(',', '.', regex=False)
df_consolidado['INDE'] = pd.to_numeric(df_consolidado['INDE'])
df_consolidado['INDE'] = df_consolidado['INDE'].round(2)

print("Tipo de dado da coluna 'INDE' após conversão:")
print(df_consolidado['INDE'].dtype)
print("\nPrimeiras 5 linhas da coluna 'INDE' após conversão:")
print(df_consolidado['INDE'].head())

In [ ]:
print("Valores nulos por coluna no df_consolidado:")
df_consolidado.isnull().sum()

In [ ]:
colunas_para_remover = ['CG', 'CF', 'CT','RECPSICOLOGIA','INDICADO','ATINGIUPV']
df_consolidado = df_consolidado.drop(columns=colunas_para_remover, errors='ignore')

print(f"Colunas {colunas_para_remover} removidas do df_consolidado.")
print("Novas colunas do df_consolidado:")
print(df_consolidado.columns.tolist())

Colunas ['CG', 'CF', 'CT', 'RECPSICOLOGIA', 'INDICADO', 'ATINGIUPV'] removidas do df_consolidado.
Novas colunas do df_consolidado:
['RA', 'TURMA', 'GENERO', 'ANOINGRESSO', 'INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'MAT', 'POR', 'ING', 'IPV', 'IAN', 'FASEIDEAL', 'DEFASAGEM', 'FASE_ESCOLAR', 'IDADEFINAL', 'PEDRAANO', 'TIPOINSTITUICAO', 'IPP', 'ANO']


In [ ]:
nome_arquivo_consolidado = 'BASE_CONSOLIDADA.xlsx'
df_consolidado.to_excel(nome_arquivo_consolidado, index=False)
print(f"DataFrame consolidado exportado com sucesso para '{nome_arquivo_consolidado}'.")